# Getting Started
This tutorial demonstrates the configuration and use of a simple BSK-RL environment.
BSK-RL and dependencies should already be installed at this point (see [Installation](../install.rst)
if you haven't installed the package yet).

## Load Modules
In this tutorial, the environment will be created with `gym.make`, so it is necessary to
import the top-level `bsk_rl` module as well as `gym` and `bsk_rl` components.

In [1]:
from typing import ClassVar

import gymnasium as gym
import numpy as np
from Basilisk.architecture import bskLogging

from bsk_rl import act, data, obs, sats, scene
from bsk_rl.sim import dyn, fsw

bskLogging.setDefaultLogLevel(bskLogging.BSK_WARNING)


If no errors were raised, you have a functional installation of `bsk_rl`.

## Configure the Satellite
[Satellites](../api_reference/sats/index.rst) are configurable agents in the environment.
To make a new environment, start by specifying the [observations](../api_reference/obs/index.rst)
and [actions](../api_reference/act/index.rst) of a satellite type, as well as the underlying
Basilisk [simulation](../api_reference/sim/index.rst) models used by the satellite.

In [2]:
class MyScanningSatellite(sats.AccessSatellite):
    observation_spec: ClassVar[list[obs.Observation]] = [
        obs.SatProperties(
            dict(prop="storage_level_fraction"),
            dict(prop="battery_charge_fraction")
        ),
        obs.Eclipse(),
    ]
    action_spec: ClassVar[list[act.Action]] = [
        act.Scan(duration=60.0),  # Scan for 1 minute
        act.Charge(duration=600.0),  # Charge for 10 minutes
    ]
    dyn_type = dyn.ContinuousImagingDynModel
    fsw_type = fsw.ContinuousImagingFSWModel

Based on this class specification, a list of configurable parameters for the satellite
can be generated.

In [3]:
MyScanningSatellite.default_sat_args()

{'hs_min': 0.0,
 'maxCounterValue': 4,
 'thrMinFireTime': 0.02,
 'desatAttitude': 'sun',
 'controlAxes_B': [1, 0, 0, 0, 1, 0, 0, 0, 1],
 'thrForceSign': 1,
 'K': 7.0,
 'Ki': -1,
 'P': 35.0,
 'imageAttErrorRequirement': 0.01,
 'imageRateErrorRequirement': None,
 'inst_pHat_B': [0, 0, 1],
 'utc_init': 'this value will be set by the world model',
 'batteryStorageCapacity': 288000.0,
 'storedCharge_Init': <function bsk_rl.sim.dyn.base.BasicDynamicsModel.<lambda>()>,
 'disturbance_vector': None,
 'dragCoeff': 2.2,
 'panelArea': 1.0,
 'imageTargetMaximumRange': -1,
 'instrumentBaudRate': 8000000.0,
 'instrumentPowerDraw': -30.0,
 'basePowerDraw': 0.0,
 'wheelSpeeds': <function bsk_rl.sim.dyn.base.BasicDynamicsModel.<lambda>()>,
 'maxWheelSpeed': inf,
 'u_max': 0.2,
 'rwBasePower': 0.4,
 'rwMechToElecEfficiency': 0.0,
 'rwElecToMechEfficiency': 0.5,
 'panelEfficiency': 0.2,
 'nHat_B': array([ 0,  0, -1]),
 'mass': 330,
 'width': 1.38,
 'depth': 1.04,
 'height': 1.58,
 'sigma_init': <function 

When instantiating a satellite, these parameters can be overriden with a constant or 
rerandomized every time the environment is reset using the ``sat_args`` dictionary.

In [4]:
sat_args = {}

# Set some parameters as constants
sat_args["imageAttErrorRequirement"] = 0.05
sat_args["dataStorageCapacity"] = 1e10
sat_args["instrumentBaudRate"] = 1e7
sat_args["storedCharge_Init"] = 50000.0

# Randomize the initial storage level on every reset
sat_args["storageInit"] = lambda: np.random.uniform(0.25, 0.75) * 1e10

# Make the satellite
sat = MyScanningSatellite(name="EO1", sat_args=sat_args)

## Making the Environment
For this example, we will be using the single-agent [SatelliteTasking](../api_reference/index.rst) 
environment. Along with passing the satellite that we configured, the environment takes
a [scenario](../api_reference/scene/index.rst), which defines the environment the
satellite is acting in, and a [rewarder](../api_reference/data/index.rst), which defines
how data collected from the scenario is rewarded.

In [5]:
env = gym.make(
    "SatelliteTasking-v1",
    satellite=sat,
    scenario=scene.UniformNadirScanning(),
    rewarder=data.ScanningTimeReward(),
    time_limit=5700.0,  # approximately 1 orbit
    log_level="INFO",
)

2026-07-28 23:40:15,813 gym                            INFO       Calling env.reset() to get observation space


2026-07-28 23:40:15,814 gym                            INFO       Resetting environment with seed=3632165663


2026-07-28 23:40:15,863 sats.satellite.EO1             INFO       <0.00> EO1: Finding opportunity windows from 0.00 to 6000.00 seconds


2026-07-28 23:40:15,872 gym                            INFO       <0.00> Environment reset


## Interacting with the Environment

First, the environment is reset.

In [6]:
observation, info = env.reset(seed=1)

2026-07-28 23:40:15,877 gym                            INFO       Resetting environment with seed=1


2026-07-28 23:40:15,892 sats.satellite.EO1             INFO       <0.00> EO1: Finding opportunity windows from 0.00 to 6000.00 seconds


2026-07-28 23:40:15,899 gym                            INFO       <0.00> Environment reset


Next, we take the scan action (`action=0`) a few times. This allows for the satellite to
settle its attitude in the nadir pointing mode to satisfy imaging conditions. Note that 
the logs show little or no data accumulated in the first two steps as it settles, but
achieves 60 reward (corresponding to 60 seconds of imaging) by the third step.

In [7]:
print("Initial data level:", observation[0], "(randomized by sat_args)")
for _ in range(3):
    observation, reward, terminated, truncated, info = env.step(action=0)
print("  Final data level:", observation[0])

2026-07-28 23:40:15,903 gym                            INFO       <0.00> === STARTING STEP ===


2026-07-28 23:40:15,904 sats.satellite.EO1             INFO       <0.00> EO1: action_nadir_scan tasked for 60.0 seconds


2026-07-28 23:40:15,904 sats.satellite.EO1             INFO       <0.00> EO1: setting timed terminal event at 60.0


2026-07-28 23:40:15,908 sats.satellite.EO1             INFO       <60.00> EO1: timed termination at 60.0 for action_nadir_scan


2026-07-28 23:40:15,909 data.base                      INFO       <60.00> Total reward: {}


...

2026-07-28 23:40:15,924 data.base                      INFO       <180.00> Total reward: {'EO1': 60.0}


2026-07-28 23:40:15,925 comm.communication             INFO       <180.00> Optimizing data communication between all pairs of satellites


2026-07-28 23:40:15,925 sats.satellite.EO1             INFO       <180.00> EO1: Satellite EO1 requires retasking


2026-07-28 23:40:15,926 gym                            INFO       <180.00> Step reward: 60.0


Initial data level: 0.5961613078 (randomized by sat_args)
  Final data level: 0.6801613078


The observation reflects the increase in stored data. The first element, corresponding
to `storage_level_fraction`, starts at a random value set by the `storageInit` function
in `sat_args` and increases based on the time spent imaging.

Finally, the charging mode is tasked repeatedly in 10-minute increments until the
environment time limit is reached.

In [8]:
while not truncated:
    observation, reward, terminated, truncated, info = env.step(action=1)
    print(f"Charge level: {observation[1]:.3f} ({env.unwrapped.simulator.sim_time:.1f} seconds)\n\tEclipse: start: {observation[2]:.1f} end: {observation[3]:.1f}")

2026-07-28 23:40:15,931 gym                            INFO       <180.00> === STARTING STEP ===


2026-07-28 23:40:15,931 sats.satellite.EO1             INFO       <180.00> EO1: action_charge tasked for 600.0 seconds


2026-07-28 23:40:15,932 sats.satellite.EO1             INFO       <180.00> EO1: setting timed terminal event at 780.0


2026-07-28 23:40:15,956 sats.satellite.EO1             INFO       <780.00> EO1: timed termination at 780.0 for action_charge


2026-07-28 23:40:15,957 data.base                      INFO       <780.00> Total reward: {}


...

2026-07-28 23:40:16,193 comm.communication             INFO       <5700.00> Optimizing data communication between all pairs of satellites


2026-07-28 23:40:16,194 gym                            INFO       <5700.00> Step reward: 0.0


2026-07-28 23:40:16,194 gym                            INFO       <5700.00> Episode terminated: False


2026-07-28 23:40:16,195 gym                            INFO       <5700.00> Episode truncated: True


Charge level: 1.000 (5580.0 seconds)
	Eclipse: start: 810.0 end: 2910.0
Charge level: 1.000 (5700.0 seconds)
	Eclipse: start: 690.0 end: 2790.0


It is observed that the battery decrease while the satellite is in eclipse, but once the
satellite is out of eclipse, the battery quickly increases to full charge.